# Packaging an empirical paper as a reproducible replication package

The core conclusion of an applied microeconomics paper often amounts to a single coefficient—a policy reduced firm outcomes by approximately 0.7. But journals (and any serious reader) do not want to see just this one number; they want the entire scaffolding around it: whether treatment and control groups are balanced before the intervention (balance table), how the baseline model is specified (two-way fixed effects), whether the coefficient remains stable across alternative control sets or standard error specifications (robustness matrix), and most critically—a script that anyone with the data can run on their own machine to reproduce the exact same result. This ensemble of outputs—balance table → baseline → robustness → runnable script—constitutes a replication package, and it is the pipeline this tutorial follows.

The methodological backbone of this pipeline is **two-way fixed effects (TWFE)** difference-in-differences: use `y ~ treatment + unit fixed effects + time fixed effects` to absorb "unit-specific differences invariant to time" and "time shocks common across all units," isolating the policy effect from confounding; cluster standard errors at the unit level to allow for within-firm correlation across years. The assumption that makes this work is still parallel trends—if there were no policy, the treatment group would evolve along a parallel trajectory to the control group—so the first substantive step in the replication package is to test for pre-trends. If pre-trends fail, the downstream coefficient is only correlation, not causation. This workflow benchmarks against R's `fixest` (`feols` for high-dimensional fixed effects + clustered SE + `etable` for tabulation) and Stata's `reghdfe` + `esttab`, following the replication standards in AER's "data & code appendix."

The data is a built-in synthetic panel: 40 firms × 8 years (2010–2017), with half the firms treated by a policy in 2015. The true treatment effect at data generation was −0.8, with clean parallel pre-trends—so we can trace every step of the replication package and bench our final coefficient estimate against the "ground truth." We run the full workflow with `socialverse`, a library designed for social-science analysis; it serves as a convenient tool here. The pipeline is: load panel → declare design → test assumptions → baseline estimate → generate replication package → plot → typeset manuscript, then inspect the evidence chain it leaves behind.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

import socialverse as sv
from socialverse import datasets as ds

# 让产物(图/稿件)始终落在本 notebook 同目录,无论从哪个工作目录运行 ——
# 这样 markdown 里的 ![](fig_xxx.png) 引用永远解析得到。
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Jupyter 里没有 __file__,退回当前目录
    _HERE = os.getcwd()

def here(fname):
    return os.path.join(_HERE, fname)

## Load panel data

We use the built-in synthetic panel `load_did_panel(att=-0.8)`. It is in long format (one row per firm × year): `firm_id` is the unit, `year` is the time index, `treat_post` marks observations that are treated, `first_treated` records the year each firm was first treated, `y` is the outcome variable, and `x1` is one covariate. Altogether, 40 firms × 8 years = 320 rows.

In [2]:
df = ds.load_did_panel(att=-0.8)
print("面板维度:", df.shape, "  (units × periods =", df.firm_id.nunique(), "×", df.year.nunique(), ")")
df.head(10)

面板维度: (320, 8)   (units × periods = 40 × 8 )


,firm_id,year,treat,post,treat_post,first_treated,y,x1
0,0,2010,1,0,0,2015,1.6221,1.5139
1,0,2011,1,0,0,2015,2.8355,0.7813
2,0,2012,1,0,0,2015,2.5744,-0.3139
3,0,2013,1,0,0,2015,3.2232,1.9603
4,0,2014,1,0,0,2015,3.5321,1.3151
5,0,2015,1,1,1,2015,2.3258,-1.2083
6,0,2016,1,1,1,2015,2.3525,0.6565
7,0,2017,1,1,1,2015,2.0104,0.3951
8,1,2010,1,0,0,2015,2.0398,0.6960
9,1,2011,1,0,0,2015,1.5657,-0.6617


## Declare research intent and design

The first step in a replication package is not to run a regression but to clarify "what does the researcher want to estimate, and what role does each column play?" Some quantities cannot be computed from data alone and must be given by the research question: the estimand is **average treatment effect on the treated (ATT)** (not mere correlation), the outcome variable is `y`, and the identification strategy is **difference-in-differences**. Recording these in the research state is how we declare research intent.

In [3]:
st = sv.StudyState()
st.write("estimand", "target", "ATT")          # 目标量:平均处理效应,不是相关
st.write("variables", "outcome", "y")          # 结果变量
st.write("identification", "strategy", "DiD")  # 识别策略:双重差分

Next, register the data and declare the design columns—which column is the unit, which is time, which is the treatment indicator, and which is the time of first treatment. `declare_design` takes only the column name strings and uses the data to verify that these columns exist; every downstream estimation function reads the design from here, so there is no need to pass parameters repeatedly.

In [4]:
sv.pp.ingest(st, data=df, name="policy_panel")
sv.pp.declare_design(
    st,
    panel_id="firm_id",
    time="year",
    treatment="treat_post",
    first_treated="first_treated",
)
st.design

Slot(panel_id, time, treatment, first_treated)

## Test parallel trends

This is the threshold for the entire replication package. The causal interpretation of DiD rests entirely on parallel trends: if there were no policy, the treatment and control groups would evolve along parallel paths. This assumption cannot be directly tested, but we can indirectly examine it using pre-treatment periods. `parallel_trends` fits a full event-study regression (unit fixed effects + time fixed effects) and then conducts a joint Wald test on all **pre-treatment** period coefficients, with the null that "all pre-treatment coefficients are zero." If `p > 0.05`, we fail to reject parallel trends and the identification assumption holds; if `p` is small, pre-trends have already diverged, and we should not interpret the downstream coefficient as causal even if it is precisely estimated.

In [5]:
sv.tl.parallel_trends(st)

pt = st.diagnostics["pretrend"]
print("平行趋势判定:", st.identification["parallel_trends"])
print(f"联合 Wald:  F = {pt['joint_F']:.3f}   p = {pt['p_value']:.3f}")
print("判定说明:", pt["note"])
print("\n各前导期系数 (相对时点 → coef, se):")
for k, (coef, se) in pt["pre_coefs"].items():
    print(f"  t={k:>3}:  {coef:+.4f}  (se {se:.4f})")

平行趋势判定: pass
联合 Wald:  F = 0.474   p = 0.755
判定说明: 未拒绝平行趋势(p>0.05)

各前导期系数 (相对时点 → coef, se):
  t= -5:  +0.0236  (se 0.1764)
  t= -4:  -0.1120  (se 0.1653)
  t= -3:  -0.2264  (se 0.1884)
  t= -2:  -0.1447  (se 0.2262)


`p = 0.755`, the four pre-treatment period coefficients are all insignificant—**we fail to reject parallel trends**. The identification assumption holds, and we can interpret the following DiD estimate as causal.

## Estimate baseline ATT

Now we fit the baseline model. `did` fits `y ~ treat_post + unit fixed effects + time fixed effects` with standard errors clustered at `firm_id` (inference on treatment effects typically requires clustering at the unit level). It also incorporates the parallel trends judgment from the previous step into the conclusions: if it passes, the result is labeled "causal ATT"; if it fails, it is downgraded to "association, not causation."

In [6]:
sv.tl.did(st)

m = st.models["did"]
print(f"ATT   = {m['att']:+.4f}")
print(f"95%CI = [{m['ci'][0]:+.4f}, {m['ci'][1]:+.4f}]")
print(f"SE    = {m['se']:.4f}   (聚类于 {m['n_clusters']} 家企业)")
print(f"p     = {m['p']:.2e}   N = {m['n']}   估计量 = {m['estimator']}")
print("结论  :", m["note"])

ATT   = -0.7307
95%CI = [-0.9308, -0.5307]
SE    = 0.1021   (聚类于 40 家企业)
p     = 8.15e-13   N = 320   估计量 = twfe_ols_cluster
结论  : 因果 ATT(平行趋势通过)


Baseline ATT ≈ **−0.731**, 95% confidence interval [−0.931, −0.531] lies entirely to the left of zero and encompasses the true value of −0.8 set at data generation—a close recovery. The policy significantly reduced the outcome variable.

## Generate replication package: balance table, robustness matrix, runnable script

The baseline coefficient is only the center of the replication package. `replicate` generates in one call the remaining pieces that reviewers want: a **balance table** comparing treatment vs. control, a **robustness matrix** showing whether point estimates are stable across specifications, a publication-quality regression table, and **exportable `main.R` (feols) and `main.do` (reghdfe) scripts that actually run**. It reads from the design and data we have already populated and deposits all these products into the research state in a single call.

In [7]:
sv.tl.replicate(st)
print("已生成产物:", "平衡表、稳健性矩阵、出版级回归表、main.R、main.do")

已生成产物: 平衡表、稳健性矩阵、出版级回归表、main.R、main.do


### Balance table: Are treatment and control comparable?

The balance table compares means of each covariate between treatment and control and reports the Imbens–Rubin **standardized difference** (`norm_diff`); values with absolute value > 0.25 are flagged in red. Here the large `norm_diff` for `treat` and `post` is **expected**—they are components of the treatment definition, so the treatment group is all 1s by construction; the covariate that matters is `x1`, with `norm_diff ≈ 0.10` well below the threshold, indicating the two groups are highly balanced on it.

In [8]:
st.diagnostics["balance"]

,treated_mean,control_mean,diff,norm_diff,flag
treat,1.00000,0.384615,0.615385,1.785411,True
post,1.00000,0.230769,0.769231,2.577019,True
x1,0.10794,0.009750,0.098190,0.096812,False


### Robustness matrix: Does the coefficient remain stable under alternative specifications?

The robustness matrix re-estimates the treatment effect across a grid of specifications: no fixed effects, fixed effects but no controls, partial controls, full controls, and with heteroskedasticity-robust standard errors. Reviewers want to see whether the point estimate is stable across this grid.

In [9]:
st.diagnostics["robustness"][["spec", "coef", "se", "stars", "n", "se_kind", "backend"]]

,spec,coef,se,stars,n,se_kind,backend
0,"(1) no FE, no controls",-0.436111,0.135934,***,320,HC1,statsmodels
1,"(2) TWFE, no controls",-0.730739,0.102079,***,320,cluster(firm_id),statsmodels
2,"(3) TWFE, half controls",-0.730739,0.102268,***,320,cluster(firm_id),statsmodels
3,"(4) TWFE, full controls",-0.732783,0.102802,***,320,cluster(firm_id),statsmodels
4,"(5) TWFE, full, robust SE",-0.732783,0.093123,***,320,HC1,statsmodels


Reading this matrix: specifications (2)–(5) all include two-way fixed effects, and ATT hovers near **−0.73**, only (1) "no fixed effects, no controls" shifts to −0.44—indicating that fixed effects are indeed absorbing confounding. All specifications are `***` (p < 0.01), and the point estimate is insensitive to whether controls are added or how standard errors are clustered. This is exactly the robustness evidence reviewers look for.

### Publication-quality regression table

The same set of specifications is formatted as a publication-ready table: coefficients with significance stars, standard errors in parentheses, and notation for which SE type is used in each column. This is the table you can paste directly into a paper.

In [10]:
st.artifacts["tables"]["regression"]

,coef,se,N,SE
specification,,,,
"(1) no FE, no controls",-0.436***,(0.136),320,HC1
"(2) TWFE, no controls",-0.731***,(0.102),320,cluster(firm_id)
"(3) TWFE, half controls",-0.731***,(0.102),320,cluster(firm_id)
"(4) TWFE, full controls",-0.733***,(0.103),320,cluster(firm_id)
"(5) TWFE, full, robust SE",-0.733***,(0.093),320,HC1


## Export reproducible scripts

The soul of a replication package is not a table but working code. `replicate` generates not placeholders but actual, syntactically correct `feols` / `reghdfe` scripts with variable names filled in; shared alongside `data.csv`, anyone can reproduce the same estimates in R or Stata. First, the R version:

In [11]:
scripts = st.artifacts["scripts"]
print(scripts["main.R"])

## AER-style replication — auto-emitted by socialverse (feols/fixest)
library(fixest)
library(modelsummary)

df <- read.csv("data.csv")

## (1) baseline TWFE
m1 <- feols(y ~ treat_post + treat + post + x1 | firm_id + year, data = df, cluster = ~firm_id)

## (2) robustness: drop controls
m2 <- feols(y ~ treat_post | firm_id + year, data = df, cluster = ~firm_id)

## (3) robustness: heteroskedasticity-robust SE
m3 <- feols(y ~ treat_post + treat + post + x1 | firm_id + year, data = df, vcov = "hetero")

## publication table
etable(m1, m2, m3, tex = FALSE,
       dict = c(treat_post = "Treatment"),
       title = "Replication: effect of treat_post on y")



One honest detail worth noting: in automatic mode without a codebook, the procedure treats `treat` / `post` / `x1` as numeric control variables and includes them in `feols(y ~ treat_post + treat + post + x1 | firm_id + year, ...)`. This is the default behavior when no codebook exists—"auto-select controls based on data type"—and because the script is **readable and exported**, you can see it at a glance and edit it as needed. This is more transparent than a black-box regression hidden inside a function. Below is the Stata version:

In [12]:
print(scripts["main.do"])

* AER-style replication — auto-emitted by socialverse (reghdfe)
import delimited "data.csv", clear
reghdfe y treat_post treat post x1, absorb(firm_id year) vce(cluster firm_id)
estimates store m1
reghdfe y treat_post, absorb(firm_id year) vce(cluster firm_id)
estimates store m2
esttab m1 m2, se star(* 0.10 ** 0.05 *** 0.01) title("Replication of treat_post on y")



## Visualization: ATT forest plot

Once results are ready, `forest` reads the point estimates and confidence intervals from `models.did` in the research state, plots them, and saves as a true PNG with the path written back to the research state.

In [13]:
sv.pl.forest(st, out=here("fig_forest_att.png"), title="Replication ATT · forest (95% CI)")
fig_info = st.artifacts["figures"]["forest"]
print("森林图已保存:", os.path.basename(fig_info["path"]), f"(dpi={fig_info['dpi']})")

森林图已保存: fig_forest_att.png (dpi=200)


A point at −0.73 with confidence interval wholly to the left of zero—a significant, robust negative treatment effect:

![Replication ATT forest plot](fig_forest_att.png)

## Typeset manuscript and structural audit

A replication package usually includes a manuscript. `manuscript_docx` handles structured formatting only—never rewrites your prose—and generates a **structural audit checklist** (section/figure/table counts, formula safety markers). If `python-docx` is installed, it writes a true `.docx`; otherwise it degrades to `.md` without losing content.

In [14]:
manuscript = (
    "# Replication: the effect of the policy on firm outcomes\n\n"
    "## 方法\n\n"
    "We estimate a two-way fixed-effects difference-in-differences model, "
    "clustering standard errors at the firm level. Parallel pre-trends are not rejected.\n\n"
    "## 结果\n\n"
    "The ATT is negative (about -0.73) and stable across the robustness matrix; "
    "every specification is significant at the 1% level.\n\n"
    "## 讨论\n\n"
    "The estimated effect is robust to the choice of controls and SE clustering."
)

sv.pl.manuscript_docx(st, manuscript=manuscript, out=here("replication_manuscript.docx"))
cov = st.diagnostics["coverage"]
print("稿件已生成:", os.path.basename(st.artifacts["docx"]), " 渲染器:", cov["renderer"])
print("必备章节覆盖:", cov["present_required"], " 缺失:", cov["missing_required"])
print("公式安全:", cov["math_note"], " 结构 OK:", cov["structure_ok"])

稿件已生成: replication_manuscript.docx  渲染器: python-docx
必备章节覆盖: ['方法', '结果', '讨论']  缺失: ['引言']
公式安全: 未检测到需转换的公式  结构 OK: True


The audit checklist honestly reports: the manuscript has "Methods / Results / Discussion" sections but lacks an "Introduction"—exactly the kind of structural feedback useful before submission.

## Reproducible evidence chain

Finally, observe the key difference between `socialverse` and ordinary replication scripts. Running the full pipeline, the research state automatically accumulates a **provenance ledger**: which function was used at each step, what it consumed, what it produced. For a replication package, "which step and data source the conclusion came from" often matters as much as the conclusion itself—this ledger gives the package an auditable supply chain.

In [15]:
print(st.summary())

StudyState
  sources: datasets, dataset_name, dataset_meta
  design: panel_id, time, treatment, first_treated
  variables: outcome, controls
  estimand: target
  identification: strategy, parallel_trends
  models: twfe, did
  diagnostics: pretrend, robustness, balance, coverage
  artifacts: scripts, tables, figures, docx
  provenance: 7 step(s)


## Summary

We packaged an empirical paper into a complete replication package: declare design → test parallel trends → baseline TWFE → balance table + robustness matrix + publication table → export R/Stata scripts → forest plot → typeset manuscript. It benchmarks against R's `fixest` (`feols`: high-dimensional fixed effects + clustered SE + `etable`) and Stata's `reghdfe` + `esttab`, plus the full AER-style paper replication protocol.

Compared with estimation tools alone, this adds two elements: parallel trends become a **hard gate that actually stops you** (failure automatically downgrades the baseline conclusion rather than silently handing you a coefficient), and an evidence chain that runs throughout and ships with the package. The next tutorial [03_complex_survey](03_complex_survey.ipynb) turns to design-weighted inference in complex-sample surveys.